# S11 · leakdepth — read-only diagnostic of the NB5 lineage negative-control failures (NB6)

Runs the protocol pre-registered as amendment **A13** before this notebook was built. **Descriptive only**: nothing is
re-tuned, the matcher is not changed, and no result here is a criterion for the tool. leakdepth 0.1.2 is used; its matcher
is identical to 0.1.1 (0.1.2 only fixes the Wilson interval at k = 0 and k = n).

| Cell | Item | What it does |
|---|---|---|
| 2 | Data | downloads the NB5 files (CIC-IDS2017 mirror, CSE-CIC-IDS2018 14–15 Feb, InSDN) over plain HTTPS; SHA-256 must equal NB5 |
| 3 | G, D1–D5 | reproduces the six N1–N3 directions exactly as NB5 did (gate G), then describes the aligned columns, the information A10 counted, labels, flow shape and concentration of the matched rows |
| 4 | D6 | internal redundancy of each corpus (exact duplicates; one random half found in the other) |
| 5 | D7 | interval coverage of the NB5 threshold-free results recomputed with the Wilson bug fix |
| 6 | Summary | compact JSON (paste it back) |

**Runtime.** Colab **High-RAM CPU**; roughly 1–1.5 h (downloads ≈ 1.25 GB). Drive is mounted once; no secrets or tokens are
requested. Outputs go to `MyDrive/S11-SecSplit/nb6/` as each job finishes. After a disconnect rerun cells 1–2, then the
interrupted cell; finished jobs are skipped.


In [ ]:
# 1. Setup: Drive (mounted once), embedded leakdepth 0.1.2 wheel (hash-checked), helpers. No secrets or tokens are used.
import base64, hashlib, json, os, subprocess, sys, time, glob, gc, re, importlib, urllib.request
LOCAL = os.environ.get("NB6_LOCAL")                     # local smoke tests only; unset on Colab
SMOKE = bool(os.environ.get("NB6_SMOKE"))
if LOCAL:
    BASE = LOCAL
else:
    if not os.path.isdir("/content/drive/MyDrive"):     # mount once; re-running cells never re-prompts
        from google.colab import drive
        drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
OUT = f"{BASE}/S11-SecSplit/nb6"
NB5 = f"{BASE}/S11-SecSplit/nb5"
os.makedirs(OUT, exist_ok=True)
WD = f"{LOCAL}/_wd" if LOCAL else "/content/nb6"
os.makedirs(WD, exist_ok=True)
WHEEL_SHA = "dd0d06f044cb5af3bf6a7ebed325b05aa8d2f260cd7dc7b980b2874ea44c9fb3"  # package files identical to the tested wheel 3727dfc4476f…; author metadata set to the release author
WHEEL_B64 = "..."
w = f"{WD}/leakdepth-0.1.2-py3-none-any.whl"
open(w, "wb").write(base64.b64decode(WHEEL_B64))
assert hashlib.sha256(open(w, "rb").read()).hexdigest() == WHEEL_SHA, "0.1.2 wheel hash mismatch"
if not LOCAL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", w, "pyarrow"], check=True)
import numpy as np, pandas as pd, leakdepth as L
LL = importlib.import_module("leakdepth.lineage")
from leakdepth.metrics import wilson
assert L.__version__ == "0.1.2", L.__version__
print("leakdepth", L.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| CPUs", os.cpu_count(),
      "| RAM GB", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))
T0 = time.time()
SEED, SAMPLE_N, TAU, N_BANDS, CHUNK, MARGIN, MAX_COST, MIN_ALIGNED = 0, 200_000, 0.8, 8, 200_000, 20.0, 0.1, 4

def done(name):
    return os.path.exists(f"{OUT}/{name}")

def jdump(obj, name):
    tmp = f"{OUT}/{name}.tmp"
    json.dump(obj, open(tmp, "w"), default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, f"{OUT}/{name}")

def jload(name):
    return json.load(open(f"{OUT}/{name}"))

def key(name):                                          # normalised column name for role lookup
    return re.sub(r"[^a-z]", "", str(name).lower())

ROLE = {"label": {"label", "attack", "attacktype", "class", "category"},
        "fwd": {"totalfwdpackets", "totfwdpkts", "totalfwdpacket", "fwdpackets", "totalforwardpackets"},
        "bwd": {"totalbackwardpackets", "totbwdpkts", "totalbwdpackets", "bwdpackets", "totalbackwardpacket"},
        "port": {"destinationport", "dstport", "dport"},
        "dur": {"flowduration"}}

def role_col(df, role):
    for c in df.columns:
        if key(c) in ROLE[role]:
            return c
    return None

def family(lbl):                                        # coarse attack family (fixed in A13 before running)
    s = str(lbl).lower()
    if re.search(r"benign|normal", s): return "benign"
    if re.search(r"web|xss|sql", s): return "web"
    if "ddos" in s: return "ddos"
    if re.search(r"slowloris|slowhttp", s): return "dos-slow"
    if re.search(r"hulk|goldeneye|dos", s): return "dos-other"
    if re.search(r"ftp|ssh|patator|bfa|brute", s): return "brute"
    if re.search(r"portscan|probe|scan", s): return "scan"
    if "bot" in s: return "bot"
    return "other"


In [ ]:
# 2. Data: the NB5 files over plain HTTPS (no token); SHA-256 must equal the NB5 record, else STOP
FILES = {"cic2017": [("rdpahalavan/CIC-IDS2017", "main", "Network-Flows/CICIDS_Flow.parquet",
                      "e64f6adfbcb7f96b018cf36c731be64ef4de3353ccd7340b190590734a0dda35")],
         "cic2018": [("c01dsnap/CIC-IDS2018", "236baa10495af451de38485123a90f60e7632bbd",
                      "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
                      "acff8bc61376ee031d80878ee6099e0b1a87a1bd711d8068298421418c9f8147"),
                     ("c01dsnap/CIC-IDS2018", "236baa10495af451de38485123a90f60e7632bbd",
                      "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv",
                      "fa2947a8256d81ee9103ae16139d62d0e17aa23e696ee80d9e76fb51c01c9c4b")],
         "insdn": [("Sharukesh/INSDN", "f59330c8ff0f1b231b382e05fc41b1e8fb0df6b4", "Dataset.csv",
                    "a93d51e1c2debc7246728578f22d84a442f156628be64b8c9bae5ccc5dd88a42")]}

def fetch(repo, rev, f, sha):
    p = f"/content/data/{repo.replace('/', '__')}/{f}"
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not (os.path.exists(p) and hashlib.sha256(open(p, "rb").read()).hexdigest() == sha):
        url = f"https://huggingface.co/datasets/{repo}/resolve/{rev}/{f}"
        with urllib.request.urlopen(url) as r, open(p + ".part", "wb") as fh:
            while True:
                b = r.read(1 << 22)
                if not b:
                    break
                fh.write(b)
        os.replace(p + ".part", p)
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    assert h == sha, f"STOP: {repo}/{f} differs from the NB5 file ({h})"
    return p

def load(name):                                         # identical reading to NB5
    if LOCAL:
        return pd.read_parquet(f"{BASE}/smoke_{name}.parquet")
    parts = []
    for repo, rev, f, sha in FILES[name]:
        p = fetch(repo, rev, f, sha)
        parts.append(pd.read_parquet(p) if f.endswith(".parquet") else pd.read_csv(p, low_memory=False))
    return pd.concat(parts, ignore_index=True)

D = {}
for nm in ("cic2017", "cic2018", "insdn"):
    D[nm] = load(nm)
    if SMOKE:
        D[nm] = D[nm].sample(n=min(20000, len(D[nm])), random_state=0).reset_index(drop=True)
    print(nm, D[nm].shape, "| label:", role_col(D[nm], "label"), "| fwd/bwd:", role_col(D[nm], "fwd"),
          role_col(D[nm], "bwd"), "| port:", role_col(D[nm], "port"), "| duration:", role_col(D[nm], "dur"))


In [ ]:
# 3. Gate G and D1–D5 for the six N1–N3 directions, reproducing NB5's lineage() steps exactly (A10, 20 bits, seed 0)
EXPECT = {"cic2017 in cic2018": 11855, "cic2018 in cic2017": 43517, "insdn in cic2017": 647,
          "cic2017 in insdn": 4232, "insdn in cic2018": 435, "cic2018 in insdn": 5606}
FAILING = ["cic2017 in cic2018", "cic2018 in cic2017", "cic2017 in insdn", "cic2018 in insdn"]
PAIRS = [("N1", "cic2017", "cic2018"), ("N2", "insdn", "cic2017"), ("N3", "insdn", "cic2018")]
QS = [0.05, 0.25, 0.5, 0.75, 0.95]

def quant(v):
    v = np.asarray(v, dtype=float); v = v[np.isfinite(v)]
    return [round(float(x), 3) for x in np.quantile(v, QS)] if len(v) else None

def share_top(s, k=10):
    vc = s.astype(str).value_counts(normalize=True)
    return {str(i): round(float(v), 4) for i, v in vc.head(k).items()}

def num(s):
    return pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)

def dup_groups(y, b_cols, kinds, decs, order):
    ys = y.sample(n=min(SAMPLE_N, len(y)), random_state=SEED)
    H = LL.canonicalise(ys, b_cols, kinds, decs).col_hashes()
    k = H.shape[1]; parent = list(range(k))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(k):
        for j in range(i + 1, k):
            if np.mean(H[:, i] == H[:, j]) >= 0.999:
                parent[find(j)] = find(i)
    roots = [find(i) for i in range(k)]                 # canonical column order
    gid = {r: g for g, r in enumerate(dict.fromkeys(roots))}
    return np.array([gid[r] for r in roots])

def describe(pid, nx, ny, x, y, px, py, al):
    sx = (x.sample(n=min(SAMPLE_N, len(x)), random_state=SEED) if len(x) > SAMPLE_N else x).reset_index(drop=True)
    res, used = LL._containment(sx, y, px, py, al, TAU, N_BANDS, SEED, CHUNK, min_aligned=MIN_ALIGNED, info_margin=MARGIN)
    d = f"{nx} in {ny}"
    got = int(res["matched"].sum())
    out = {"pair": pid, "direction": d, "sampled_rows": len(sx), "matched": got, "expected_nb5": EXPECT.get(d),
           "gate_G_reproduced": got == EXPECT.get(d), "rejected_low_info": int(res["rejected_low_info"].sum()),
           "info_needed_bits": round(float(res["info_needed"]), 2), "aligned": len(used),
           "stages": pd.Series([u[3] for u in used]).value_counts().to_dict()}
    a_cols = [u[0] for u in used]; b_cols = [u[1] for u in used]
    kinds = ["num" if px[c].kind == "num" else "str" for c in a_cols]
    decs = []
    for a, b, k in zip(a_cols, b_cols, kinds):
        dd = [q for q in (px[a].decimals, py[b].decimals) if q is not None] if k == "num" else []
        decs.append(min(dd) if dd else None)
    order = [i for i, k in enumerate(kinds) if k == "num"] + [i for i, k in enumerate(kinds) if k != "num"]
    mi = np.flatnonzero(res["matched"]); part = res["partner"][mi]
    groups = dup_groups(y, b_cols, kinds, decs, order)
    out["b_duplicate_groups"] = int(groups.max() + 1) if len(groups) else 0
    if got == 0:
        jdump(out, f"D_{d.replace(' ', '_')}.json"); return out
    ca = LL.canonicalise(sx.iloc[mi], a_cols, kinds, decs)
    k = ca.k
    mask = np.zeros((len(mi), k), bool); bits = np.zeros((len(mi), k))
    for c0 in range(0, len(y), CHUNK):                  # bits exactly as A10 computes them: per streamed B chunk
        sel = np.flatnonzero((part >= c0) & (part < c0 + CHUNK))
        if len(sel) == 0:
            continue
        cb = LL.canonicalise(y.iloc[c0:c0 + CHUNK], b_cols, kinds, decs)
        bm = LL.bits_matrix(cb.col_hashes())
        mask[sel] = LL.agree_mask(ca, sel, cb, part[sel] - c0)
        bits[sel] = bm[part[sel] - c0]
    mb = mask * bits
    info = mb.sum(axis=1)
    gbits = np.zeros(len(mi))
    for g in range(groups.max() + 1):
        gbits += mb[:, groups == g].max(axis=1)
    need = float(res["info_needed"])
    names = [(a_cols[i], b_cols[i], used[i][3], kinds[i], decs[i]) for i in order]
    rows = []
    for j, (ca_, cb_, how, kd, dc) in enumerate(names):
        pa_, pb_ = px[ca_], py[cb_]
        rows.append({"a_col": ca_, "b_col": cb_, "stage": how, "kind": kd, "decimals": dc,
                     "zero_a": round(pa_.zero, 4), "zero_b": round(pb_.zero, 4),
                     "distinct_a": pa_.n_distinct, "distinct_b": pb_.n_distinct, "b_dup_group": int(groups[j]),
                     "agree_matched": round(float(mask[:, j].mean()), 4),
                     "mean_bits_when_agreeing": round(float(bits[mask[:, j], j].mean()), 3) if mask[:, j].any() else None})
    D1 = pd.DataFrame(rows)
    D1.to_csv(f"{OUT}/D1_{d.replace(' ', '_')}.csv", index=False)
    out["D1_top_bits"] = D1.assign(contrib=D1["agree_matched"] * D1["mean_bits_when_agreeing"].fillna(0)) \
        .sort_values("contrib", ascending=False).head(10)[["a_col", "b_col", "stage", "agree_matched", "mean_bits_when_agreeing", "b_dup_group"]] \
        .values.tolist()
    out["D1_cols_zero_ge_0.9_in_b"] = int((D1["zero_b"] >= 0.9).sum())
    out["D2"] = {"bits_recheck_all_ge_need": bool((info >= need - 1e-9).all()), "agreeing_cols_q": quant(mask.sum(axis=1)),
                 "a10_bits_q": quant(info), "group_bits_q": quant(gbits), "ratio_group_over_a10_q": quant(gbits / info),
                 "share_ge_need_under_group_count": round(float(np.mean(gbits >= need)), 4)}
    la, lb = role_col(x, "label"), role_col(y, "label")
    if la and lb:
        A = sx[la].iloc[mi].astype(str).str.strip().to_numpy(); B = y[lb].iloc[part].astype(str).str.strip().to_numpy()
        fa = np.array([family(v) for v in A]); fb = np.array([family(v) for v in B])
        fs = sx[la].astype(str).str.strip().map(family)
        out["D3"] = {"labels_a_top15": share_top(pd.Series(A), 15), "labels_b_top15": share_top(pd.Series(B), 15),
                     "label_pairs_top20": share_top(pd.Series(A) + " -> " + pd.Series(B), 20),
                     "nonbenign_a": round(float(np.mean(fa != "benign")), 4), "nonbenign_b": round(float(np.mean(fb != "benign")), 4),
                     "same_family_nonbenign": round(float(np.mean((fa == fb) & (fa != "benign"))), 4),
                     "family_pairs": share_top(pd.Series(fa) + " -> " + pd.Series(fb), 12),
                     "baseline_nonbenign_a_sample": round(float(np.mean(fs != "benign")), 4),
                     "baseline_family_a_sample": share_top(fs, 10)}
    else:
        out["D3"] = {"labels": "label column not found", "a": la, "b": lb}
    D4 = {}
    fw, bw = role_col(x, "fwd"), role_col(x, "bwd")
    if fw and bw:
        tot = num(sx[fw]) + num(sx[bw])
        D4.update(pkts_le2_matched=round(float(np.mean(tot[mi] <= 2)), 4), pkts_le2_sample=round(float(np.mean(tot <= 2)), 4),
                  pkts_le4_matched=round(float(np.mean(tot[mi] <= 4)), 4), pkts_le4_sample=round(float(np.mean(tot <= 4)), 4))
    cfull = LL.canonicalise(sx, a_cols, kinds, decs)
    if cfull.num.shape[1]:
        zf = np.mean(cfull.num == 0, axis=1)
        D4.update(zero_share_median_matched=round(float(np.median(zf[mi])), 4), zero_share_median_sample=round(float(np.median(zf)), 4),
                  zero_share_ge_half_matched=round(float(np.mean(zf[mi] >= 0.5)), 4))
    pc, dc_ = role_col(x, "port"), role_col(x, "dur")
    if pc:
        D4.update(ports_matched_top10=share_top(sx[pc].iloc[mi]), ports_sample_top10=share_top(sx[pc]))
    if dc_:
        dur = num(sx[dc_]); D4.update(duration_q_matched=quant(dur[mi]), duration_q_sample=quant(dur))
    out["D4"] = D4
    th = pd.Series(LL._combine(ca.col_hashes(), np.arange(k)))
    vc = th.value_counts(); pp = pd.Series(part).value_counts()
    out["D5"] = {"distinct_tuples": int(len(vc)), "top10_tuple_share": round(float(vc.head(10).sum() / len(th)), 4),
                 "distinct_partners": int(len(pp)), "max_rows_per_partner": int(pp.max())}
    jdump(out, f"D_{d.replace(' ', '_')}.json")
    return out

for pid, an, bn in PAIRS:
    names = [f"{an} in {bn}", f"{bn} in {an}"]
    if all(done(f"D_{n.replace(' ', '_')}.json") for n in names):
        continue
    t0 = time.time(); a, b = D[an], D[bn]
    pa, pb = LL.profile_columns(a, SAMPLE_N, SEED), LL.profile_columns(b, SAMPLE_N, SEED)   # as lineage(): once per pair
    informative = [x for x in LL.align_columns(pa, pb, MAX_COST) if pa[x[0]].kind != "const"]
    for (nx, ny, x, y, px, py, al) in ((an, bn, a, b, pa, pb, informative),
                                       (bn, an, b, a, pb, pa, [(q, p, c, h) for p, q, c, h in informative])):
        if not done(f"D_{nx}_in_{ny}.json"):
            describe(pid, nx, ny, x, y, px, py, al)
    print(pid, "done in", round(time.time() - t0), "s"); gc.collect()

for n in EXPECT:
    if done(f"D_{n.replace(' ', '_')}.json"):
        r = jload(f"D_{n.replace(' ', '_')}.json")
        print(f"{n:22s} matched {r['matched']:6d} (NB5 {r['expected_nb5']}) G={'OK' if r['gate_G_reproduced'] else 'NOT REPRODUCED'}"
              f" | aligned {r['aligned']} | dup groups {r['b_duplicate_groups']}"
              + (f" | nonbenign A {r['D3'].get('nonbenign_a')} same-family {r['D3'].get('same_family_nonbenign')}"
                 f" | <=2 pkts {r['D4'].get('pkts_le2_matched')} | group/A10 bits median {r['D2']['ratio_group_over_a10_q'][2]}"
                 if r["matched"] else ""))


In [ ]:
# 4. D6 internal redundancy: exact duplicates on non-label columns; one random half found in the other (A10, seed 0)
for nm in ("cic2017", "cic2018", "insdn"):
    fn = f"D6_{nm}.json"
    if done(fn):
        continue
    t0 = time.time(); df = D[nm]; lc = role_col(df, "label")
    h = pd.util.hash_pandas_object(df.drop(columns=[lc] if lc else []), index=False)
    perm = np.random.default_rng(SEED).permutation(len(df)); m = len(df) // 2
    h1, h2 = df.iloc[perm[:m]].reset_index(drop=True), df.iloc[perm[m:]].reset_index(drop=True)
    rep = L.lineage(h1, h2, "half1", "half2", seed=SEED).to_dict()
    jdump({"corpus": nm, "rows": len(df), "rows_in_duplicate_groups": round(float(h.duplicated(keep=False).mean()), 4),
           "rows_beyond_first_copy": round(float(h.duplicated().mean()), 4),
           "half_in_half": [(c["direction"], c["matched"], c["sampled_rows"], round(c["containment"], 4), c["aligned_columns"])
                            for c in rep["results"]["containment"]], "sec": round(time.time() - t0)}, fn)
    del h1, h2; gc.collect()
for nm in ("cic2017", "cic2018", "insdn"):
    if done(f"D6_{nm}.json"):
        print(jload(f"D6_{nm}.json"))


In [ ]:
# 5. D7: NB5 threshold-free interval coverage recomputed with the 0.1.2 Wilson fix (only the interval edges change)
TF = f"{NB5}/T_threshold_free.jsonl"
D7 = None
if os.path.exists(TF):
    rec = []
    for r in map(json.loads, open(TF)):
        for m in ("a10", "frozen"):
            v = r.get(m, {})
            if not v.get("aligned"):
                continue
            s = r["planted"] / r["n"]; lo, hi = wilson(v["matched"], r["n"])
            rec.append({"mode": m, "grp": "noise" if r["transform"] == "noise" else "non-noise", "rho": r["rho"],
                        "old": v["lo"] <= s <= v["hi"], "new": lo <= s <= hi, "changed_lo": lo != v["lo"]})
    V = pd.DataFrame(rec)
    D7 = {f"{k[0]}|{k[1]}|{k[2]}": [round(float(g["old"].mean()), 3), round(float(g["new"].mean()), 3)]
          for k, g in V.groupby(["mode", "grp", "rho"])}
    D7["rows_with_changed_lower_bound"] = int(V["changed_lo"].sum()); D7["rows"] = len(V)
    jdump(D7, "D7_coverage.json")
    print("coverage [old, fixed] by mode|group|rho:", D7)
else:
    print("NB5 T file not found:", TF)


In [ ]:
# 6. Compact summary (paste this output back); also nb6/S11_NB6_summary.json
S = {"leakdepth": L.__version__, "G": {}, "dir": {}, "readings": {}}
for n in EXPECT:
    fn = f"D_{n.replace(' ', '_')}.json"
    if not done(fn):
        S["G"][n] = "missing"; continue
    r = jload(fn); S["G"][n] = [r["matched"], r["expected_nb5"], r["gate_G_reproduced"]]
    S["dir"][n] = {k: r.get(k) for k in ("aligned", "stages", "rejected_low_info", "info_needed_bits", "b_duplicate_groups",
                                          "D1_cols_zero_ge_0.9_in_b", "D1_top_bits", "D2", "D3", "D4", "D5")}
    if n in FAILING and r["matched"]:
        d3, d4, d2 = r["D3"], r["D4"], r["D2"]
        h1a = d3.get("nonbenign_a") is not None and d3["nonbenign_a"] >= 0.5
        S["readings"][n] = {
            "H1_attack_dominated": h1a if d3.get("nonbenign_a") is not None else "no labels",
            "H1_same_family_attack_flows": (h1a and d3["same_family_nonbenign"] >= 0.5) if d3.get("nonbenign_a") is not None else "no labels",
            "H2_short_or_zero_dominated": bool((d4.get("pkts_le2_matched") or 0) >= 0.5 or (d4.get("zero_share_median_matched") or 0) >= 0.5),
            "H3_bound_inflated_by_duplicate_columns": bool(d2["ratio_group_over_a10_q"][2] <= 0.5),
            "G_reproduced": r["gate_G_reproduced"]}
S["D6"] = {nm: jload(f"D6_{nm}.json") for nm in ("cic2017", "cic2018", "insdn") if done(f"D6_{nm}.json")}
S["D7_coverage[old,fixed]"] = jload("D7_coverage.json") if done("D7_coverage.json") else None
S["runtime_min_this_session"] = round((time.time() - T0) / 60, 1)
jdump(S, "S11_NB6_summary.json")
print(json.dumps(S, separators=(",", ":"), default=str))
